# MCP Unstructured v6.1 Test Notebook

This notebook installs dependencies, starts the server in the background, checks health, and tests `parse_file`.


In [ ]:
PROJECT_DIR = '/content/mcp_unstructured'
!apt-get update -y
!apt-get install -y poppler-utils tesseract-ocr libmagic-dev
!pip uninstall -y numpy
!pip install --no-cache-dir numpy==1.26.4
!pip install --no-cache-dir -r {PROJECT_DIR}/requirements.txt
print('IMPORTANT: Restart the runtime now, then continue with the remaining cells.')


In [ ]:
import os, sys
os.chdir('/content/mcp_unstructured')
sys.path.insert(0, os.getcwd())
os.environ['ALLOWED_ROOT'] = '/content'
os.environ['SCANNED_PDF_POLICY'] = 'ocr_only'
print('Working directory:', os.getcwd())
print('ALLOWED_ROOT =', os.environ['ALLOWED_ROOT'])
print('SCANNED_PDF_POLICY =', os.environ['SCANNED_PDF_POLICY'])


In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))


In [ ]:
import threading
import server
thread = threading.Thread(target=lambda: server.run_http(8000), daemon=True)
thread.start()
print('Server started on http://localhost:8000')


In [ ]:
import requests
resp = requests.post('http://localhost:8000', json={'tool': 'health'}, timeout=30)
print(resp.status_code)
print(resp.json())


In [ ]:
uploaded_files = [f'/content/{name}' for name in uploaded.keys()]
sample_path = uploaded_files[0]
print('Sample path:', sample_path)


In [ ]:
import requests
payload = {'tool': 'parse_file', 'path': sample_path, 'route': 'fast', 'chunking_strategy': 'basic'}
resp = requests.post('http://localhost:8000', json=payload, timeout=180)
print('status:', resp.status_code)
print(resp.text[:2000])


In [ ]:
import requests
payload = {'tool': 'parse_file', 'path': sample_path, 'route': 'auto', 'chunking_strategy': 'basic'}
resp = requests.post('http://localhost:8000', json=payload, timeout=180)
print('status:', resp.status_code)
print(resp.text[:2000])
